In [ ]:
import pandas as pd

PATH_HAINEUX  = 'corpus_brut_haineux\corpus_haineux.csv'                    
PATH_FEATURES = 'corpus_emotion_haineux_features.csv'  

df_h = pd.read_csv(PATH_HAINEUX, sep=';')
df_e = pd.read_csv(PATH_FEATURES, sep=',', index_col='file_id')


df_h['tweet_clean'] = df_h['tweet'].str.strip().str.replace(r'\s+', ' ', regex=True)
df_e['text_clean']  = df_e['text'].str.strip().str.replace(r'\s+', ' ', regex=True)


df_e = df_e.reset_index()  
print(f'corpus_haineux  : {len(df_h)} tweets')
print(f'corpus_features : {len(df_e)} phrases')
print(f'Colonnes émotion : {df_e.columns.tolist()}')

corpus_haineux  : 600 tweets
corpus_features : 793 phrases
Colonnes émotion : ['file_id', 'feature_found', 'avec_cat_joie', 'avec_cat_admiration', 'avec_cat_tristesse', 'avec_cat_surprise', 'avec_emo_comportementale', 'avec_cat_autre', 'avec_emo_designee', 'avec_emo', 'avec_emo_base', 'avec_emo_suggeree', 'avec_cat_embarras', 'avec_cat_colere', 'avec_cat_fierte', 'avec_emo_complexe', 'avec_emo_montree', 'avec_cat_peur', 'text', 'text_clean']


In [2]:
# ── Identification des colonnes binaires ─────────────────────────────────────

# Toutes les colonnes qui commencent par "avec_"
FEATURE_COLS = [c for c in df_e.columns if c.startswith('avec_')]

print(f'{len(FEATURE_COLS)} colonnes binaires :')
for c in FEATURE_COLS:
    print(f'  - {c}')

16 colonnes binaires :
  - avec_cat_joie
  - avec_cat_admiration
  - avec_cat_tristesse
  - avec_cat_surprise
  - avec_emo_comportementale
  - avec_cat_autre
  - avec_emo_designee
  - avec_emo
  - avec_emo_base
  - avec_emo_suggeree
  - avec_cat_embarras
  - avec_cat_colere
  - avec_cat_fierte
  - avec_emo_complexe
  - avec_emo_montree
  - avec_cat_peur


In [4]:
# ── Fonctions utilitaires ─────────────────────────────────────────────────────

def no_relation(te: str, tw: str) -> bool:

    return te != tw and te not in tw and tw not in te


def find_next_matching_tweet(df_h, df_e, i_h_start: int, i_e: int,
                              max_lookahead: int = 10) -> int | None:
   
    te = df_e.iloc[i_e]['text_clean']
    for k in range(max_lookahead):
        idx = i_h_start + k
        if idx >= len(df_h):
            break
        if not no_relation(te, df_h.iloc[idx]['tweet_clean']):
            return idx
    return None


def or_features(current: dict, row_e) -> dict:
  
    for col in FEATURE_COLS:
        current[col] = max(current[col], int(row_e[col]))
    return current




In [ ]:
# ── Algorithme à deux indices ─────────────────────────────────────────────────

i_h, i_e = 0, 0
n_h, n_e = len(df_h), len(df_e)

# Initialiser les accumulateurs : 0 pour chaque feature binaire
assigned = {
    i: {
        **{col: 0 for col in FEATURE_COLS},
        'feature_found': False,
        'e_ids': [],
        'cas': None
    }
    for i in range(n_h)
}

while i_h < n_h and i_e < n_e:
    tw   = df_h.iloc[i_h]['tweet_clean']
    te   = df_e.iloc[i_e]['text_clean']
    eid  = df_e.iloc[i_e]['file_id']
    has_feat = bool(df_e.iloc[i_e]['feature_found'])

  
    if te == tw:
        assigned[i_h]['cas'] = 'exact'
        assigned[i_h] = or_features(assigned[i_h], df_e.iloc[i_e])
        if has_feat:
            assigned[i_h]['feature_found'] = True
        assigned[i_h]['e_ids'].append(eid)
        i_h += 1
        i_e += 1


   
    elif te in tw:
        assigned[i_h]['cas'] = 'e_in_tw'
        assigned[i_h] = or_features(assigned[i_h], df_e.iloc[i_e])
        if has_feat:
            assigned[i_h]['feature_found'] = True
        assigned[i_h]['e_ids'].append(eid)
        i_e += 1
        # Avancer tweet si la prochaine phrase ne lui appartient plus
        if i_e < n_e and no_relation(df_e.iloc[i_e]['text_clean'], tw):
            i_h += 1

   
    
    elif tw in te:
        assigned[i_h]['cas'] = 'tw_in_e'
        assigned[i_h] = or_features(assigned[i_h], df_e.iloc[i_e])
        if has_feat:
            assigned[i_h]['feature_found'] = True
        assigned[i_h]['e_ids'].append(eid)
        i_h += 1  # PAS i_e

    # ── ORPHAN : aucune relation ──────────────────────────────────────────────
    else:
        next_h = find_next_matching_tweet(df_h, df_e, i_h, i_e, max_lookahead=10)
        if next_h is not None and next_h != i_h:
            i_h = next_h  # sauter les tweets fantômes
        else:
            i_e += 1  # phrase orpheline (chevauchement de frontière)

print(f'Terminé : i_h={i_h}/{n_h}, i_e={i_e}/{n_e}')

Terminé : i_h=600/600, i_e=793/793


In [7]:
# ── Construction du dataframe final ──────────────────────────────────────────

rows = []
for i in range(n_h):
    feat = assigned[i]
    row = {
        'tweet'         : df_h.iloc[i]['tweet'],
        'label'         : df_h.iloc[i]['label'],
        'feature_found' : feat['feature_found'],
        'emotion_ids'   : feat['e_ids'],
        'cas'           : feat['cas'] or 'no_match',
    }
    for col in FEATURE_COLS:
        row[col] = feat[col]
    rows.append(row)

df_merged = pd.DataFrame(rows)


In [ ]:



print('=== CAS 2 – e_in_tw : h[0] ===')
r = df_merged.iloc[0]
print(f'  cas           : {r["cas"]}')
print(f'  emotion_ids   : {r["emotion_ids"]}')
print(f'  feature_found : {r["feature_found"]}')
active = [c for c in FEATURE_COLS if r[c] == 1]
print(f'  features à 1  : {active}')
print()


print('=== CAS 3 – tw_in_e : h[1], h[2], h[3] ===')
for i in [1, 2, 3]:
    r = df_merged.iloc[i]
    active = [c for c in FEATURE_COLS if r[c] == 1]
    print(f'  h[{i}] cas={r["cas"]:8s} feat_found={r["feature_found"]} features_à_1={active}')
print()


df_merged[df_merged['feature_found'] == True][['tweet', 'label'] + FEATURE_COLS].head(8)

=== CAS 2 – e_in_tw : h[0] ===
  cas           : e_in_tw
  emotion_ids   : [np.int64(1), np.int64(2), np.int64(3)]
  feature_found : True
  features à 1  : ['avec_cat_surprise', 'avec_emo', 'avec_emo_base', 'avec_emo_suggeree', 'avec_emo_montree', 'avec_cat_peur']

=== CAS 3 – tw_in_e : h[1], h[2], h[3] ===
  h[1] cas=tw_in_e  feat_found=False features_à_1=[]
  h[2] cas=tw_in_e  feat_found=False features_à_1=[]
  h[3] cas=tw_in_e  feat_found=False features_à_1=[]



,tweet,label,avec_cat_joie,avec_cat_admiration,avec_cat_tristesse,avec_cat_surprise,avec_emo_comportementale,avec_cat_autre,avec_emo_designee,avec_emo,avec_emo_base,avec_emo_suggeree,avec_cat_embarras,avec_cat_colere,avec_cat_fierte,avec_emo_complexe,avec_emo_montree,avec_cat_peur
0,Tous ces migrants qui envahissent nos hôpitaux...,1,0,0,0,1,0,0,0,1,1,1,0,0,0,0,1,1
4,"oui, mais y'a soros (ami de macron, soit dit e...",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
5,En plein délire parce que 3 migrants ont fabri...,1,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0
6,@ZKongkin @MadjidFalastine @GILLESLEMAU Les mi...,1,0,1,0,0,0,0,0,1,1,1,0,1,0,0,0,0
7,@Ombrelumire2 @IsabelleChipeau Ils nous envahi...,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0
8,@BFMTV C'est plus important de s'occuper des m...,1,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0
9,@DamienRieu @GiannoniMartine @Prefet25 J'aime ...,1,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,1
10,@KimJongUnique On leur paye leTGV à ces connar...,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,1


In [ ]:


df_export = df_merged.drop(columns=['cas']).copy()
df_export['emotion_ids'] = df_export['emotion_ids'].apply(str)

df_export.to_csv('corpus_haine_emotic_merged_features.csv', sep=',', index=False, encoding='utf-8-sig')
print("Fichier 'corpus_haine_emotic_merged_features.csv' exporté (sep=',').")
df_export.head(5)

Fichier 'corpus_haine_emotic_merged_features.csv' exporté (sep=',').


,tweet,label,feature_found,emotion_ids,avec_cat_joie,avec_cat_admiration,avec_cat_tristesse,avec_cat_surprise,avec_emo_comportementale,avec_cat_autre,avec_emo_designee,avec_emo,avec_emo_base,avec_emo_suggeree,avec_cat_embarras,avec_cat_colere,avec_cat_fierte,avec_emo_complexe,avec_emo_montree,avec_cat_peur
0,Tous ces migrants qui envahissent nos hôpitaux...,1,True,"[np.int64(1), np.int64(2), np.int64(3)]",0,0,0,1,0,0,0,1,1,1,0,0,0,0,1,1
1,Les #migrants #maghrebins qui envahissent la🇫🇷...,1,False,[np.int64(4)],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,#Immigration #islamistes #racailles \n,1,False,[np.int64(4)],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,@philippe_dormoy @F_Desouche Retour à la cas...,1,False,[np.int64(4)],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,"oui, mais y'a soros (ami de macron, soit dit e...",1,True,[np.int64(5)],0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
